In [0]:
%pip install catboost==1.2.10 openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
from pathlib import Path
import sys
import json
import re

import numpy as np
import pandas as pd
import catboost
from catboost import CatBoostClassifier

# ============================================================
# DATEI EINTRAGEN
# ============================================================

BASIS = Path("/Workspace/Users/khalil-said.albert@de.abb.com")

DATEI = BASIS / "CORR_2_ORIGINAL_191007_Stuecklisten.aufgeloest (002)_korrigiert - Copy.xlsx"

SNAPSHOT = BASIS / "Daten" / "catboost_v2_snapshot_2026-09-18"
HAUPTGRUPPE = "GJL121"
FEHLT = "OHNE_ANGABE"

if not DATEI.is_file():
    raise FileNotFoundError(f"Datei nicht gefunden: {DATEI}")

MODUL_PFAD = BASIS / "Daten"
if str(MODUL_PFAD) not in sys.path:
    sys.path.insert(0, str(MODUL_PFAD))

import bom_core as bc
import layer2_profile as L2


# ============================================================
# GESPEICHERTE MODELLE UND ZUORDNUNGEN LADEN
# ============================================================

with open(SNAPSHOT / "metadaten.json", encoding="utf-8") as f:
    meta = json.load(f)

if catboost.__version__ != meta["catboost_version"]:
    raise RuntimeError(
        f"Benötigt: CatBoost {meta['catboost_version']}; "
        f"installiert: {catboost.__version__}"
    )

modell_spalten_v2 = {
    int(fold): list(spalten) for fold, spalten in meta["modellspalten"].items()
}

if set(modell_spalten_v2) != set(range(5)):
    raise ValueError("Im Snapshot fehlen Modellspalten für einen Fold.")

modelle_v2 = {}

for fold, spalten in sorted(modell_spalten_v2.items()):
    modell = CatBoostClassifier()
    modell.load_model(str(SNAPSHOT / f"catboost_v2_fold_{fold}.cbm"))

    if list(modell.feature_names_) != spalten:
        raise ValueError(f"Fold {fold}: Modell und gespeicherte Spalten passen nicht.")

    modelle_v2[fold] = modell

original = pd.read_csv(
    SNAPSHOT / "ergebnisse_v2.csv",
    dtype={"erzeugnis": str, "komponente": str},
)

original["fold"] = original["fold"].astype(int)

if not original.groupby("erzeugnis")["fold"].nunique().eq(1).all():
    raise ValueError("Widersprüchliche Fold-Zuordnung im Snapshot.")

fold_je_produkt = original.groupby("erzeugnis")["fold"].first()

# Im Main erzeugt pd.crosstab die alphabetisch sortierten Spalten.
# Diese ursprüngliche Zuordnung bleibt auch für die neue Datei bestehen.
komponenten = pd.Index(
    sorted(original["komponente"].unique()),
    name="komponente",
)

# Bewertbarkeit aus dem eingefrorenen Stand übernehmen.
original["_bewertbar"] = original["p_anwesend"].notna()

if original.groupby(["fold", "komponente"])["_bewertbar"].nunique().gt(1).any():
    raise ValueError("Widersprüchliche Bewertbarkeit im Snapshot.")

status_original = original[
    ["fold", "komponente", "pruefstatus", "_bewertbar"]
].drop_duplicates(["fold", "komponente"])

print("Fünf gespeicherte V2-Modelle geladen.")


# ============================================================
# DATEI WIE IM MAIN AUFBEREITEN
# ============================================================

sauber, protokoll = bc.clean_bom(str(DATEI))

ausgabe_clean = BASIS / "Stueckliste_bereinigt_CORRUPT.xlsx"

sauber_corrupt_df = pd.DataFrame(sauber)

sauber_corrupt_df.to_excel(
    ausgabe_clean,
    index=False
)

print("Gespeichert unter:", ausgabe_clean)
print("Bereinigte Positionszeilen:", len(sauber_corrupt_df))

quelle = pd.DataFrame(sauber)

pflicht = [
    "erzeugnis",
    "block",
    "komponente",
    "erzeugnis_txt",
    "komponente_txt",
]

if not set(pflicht).issubset(quelle.columns):
    raise ValueError(f"Spalten fehlen: {set(pflicht) - set(quelle.columns)}")

if quelle[["erzeugnis", "block", "komponente"]].isna().any().any():
    raise ValueError("Produkt, Block oder Komponente fehlt.")

quelle["erzeugnis"] = quelle["erzeugnis"].astype(str)
quelle["komponente"] = quelle["komponente"].astype(str)

block_sets = (
    quelle.groupby(["erzeugnis", "block"])["komponente"]
    .agg(lambda x: tuple(sorted(set(x))))
    .reset_index(name="komponentenset")
)

fassungen = (
    block_sets[["erzeugnis", "komponentenset"]]
    .drop_duplicates(["erzeugnis", "komponentenset"])
    .reset_index(drop=True)
)

fassungen["fassung_id"] = fassungen.index
fassungen["hauptgruppe"] = fassungen["erzeugnis"].apply(L2.hauptgruppe_von)

fassungen = fassungen.loc[fassungen["hauptgruppe"].eq(HAUPTGRUPPE)].copy()

if fassungen.empty:
    raise ValueError(f"Keine Produkte aus {HAUPTGRUPPE} gefunden.")

fassungen["produktgewicht"] = 1.0 / fassungen.groupby("erzeugnis")[
    "fassung_id"
].transform("count")

fassungen["fold"] = fassungen["erzeugnis"].map(fold_je_produkt)

if fassungen["fold"].isna().any():
    unbekannt = fassungen.loc[fassungen["fold"].isna(), "erzeugnis"].unique()
    raise ValueError(f"Produkte ohne gespeicherte Fold-Zuordnung: {unbekannt.tolist()}")

fassungen["fold"] = fassungen["fold"].astype(int)

quelle_pilot = quelle.loc[quelle["erzeugnis"].isin(fassungen["erzeugnis"])].copy()

vorhanden = (
    fassungen[["fassung_id", "komponentenset"]]
    .explode("komponentenset")
    .rename(columns={"komponentenset": "komponente"})
)

X = pd.crosstab(
    vorhanden["fassung_id"],
    vorhanden["komponente"],
).reindex(
    index=fassungen["fassung_id"],
    columns=komponenten,
    fill_value=0,
)

X = (X > 0).astype("int8")

paare = (
    X.rename_axis(index="fassung_id", columns="komponente")
    .stack()
    .rename("vorhanden")
    .reset_index()
    .merge(
        fassungen[["fassung_id", "erzeugnis", "produktgewicht", "fold"]],
        on="fassung_id",
        how="left",
        validate="many_to_one",
    )
)


# ============================================================
# MODELLMERKMALE WIE IM MAIN ERZEUGEN
# ============================================================

kontext = X.loc[paare["fassung_id"]].to_numpy(copy=True)
ziel_spalten = komponenten.get_indexer(paare["komponente"])

# Eigene Anwesenheit wie im Training ausblenden.
kontext[np.arange(len(paare)), ziel_spalten] = 0

features = pd.DataFrame(
    kontext,
    columns=[f"kontext_{i}" for i in range(len(komponenten))],
)

features["pruefkomponente"] = paare["komponente"].astype(str)


def spannung(text):
    t = str(text).replace("*", " ").replace("+", " ").replace("_", " ")
    treffer = re.findall(r"(\d+(?:[-/]\d+)?)\s*V\s*(DC|AC)?", t, re.I)
    werte = set()

    for wert, art in treffer:
        art = art.upper() or (
            "AC" if re.search(r"\d+(?:-\d+)?\s*HZ", t, re.I) else "UNBEKANNT"
        )
        werte.add(f"{wert}_{art}")

    return next(iter(werte)) if len(werte) == 1 else "MEHRDEUTIG" if werte else FEHLT


def eindeutig(spalte):
    werte = set(spalte.fillna(FEHLT))
    return next(iter(werte)) if len(werte) == 1 else "MEHRDEUTIG"


quelle_pilot["kleinere_gruppe"] = (
    quelle_pilot["erzeugnis_txt"].fillna("").map(lambda t: L2.typ_von(str(t)) or FEHLT)
)

quelle_pilot["produkt_spannung"] = quelle_pilot["erzeugnis_txt"].map(spannung)

quelle_pilot["komponente_spannung"] = quelle_pilot["komponente_txt"].map(spannung)

produkt_merkmale = quelle_pilot.groupby("erzeugnis")[
    ["kleinere_gruppe", "produkt_spannung"]
].agg(eindeutig)

komponenten_spannung = (
    quelle_pilot.groupby("komponente")["komponente_spannung"]
    .agg(eindeutig)
    .reindex(komponenten)
)

# Fehlende Komponentenmerkmale nicht unbemerkt ersetzen.
if komponenten_spannung.isna().any():
    fehlend = komponenten_spannung.index[komponenten_spannung.isna()].tolist()
    raise ValueError(
        "Diese Komponenten fehlen vollständig in der neuen Datei. "
        "Ihre ursprünglichen Spannungsmerkmale werden benötigt: "
        f"{fehlend}"
    )

komponenten_basis = (
    komponenten.to_series()
    .str.extract(r"^(.+?)[A-Za-z]\d{4}$", expand=False)
    .fillna(FEHLT)
)

for merkmal in produkt_merkmale.columns:
    features[merkmal] = paare["erzeugnis"].map(produkt_merkmale[merkmal])

features["komponente_spannung"] = paare["komponente"].map(komponenten_spannung)

features["komponente_basis"] = paare["komponente"].map(komponenten_basis)


# ============================================================
# VORHERSAGEN MIT DEN GESPEICHERTEN MODELLEN
# ============================================================

ergebnisse_neu = paare.merge(
    status_original,
    on=["fold", "komponente"],
    how="left",
    validate="many_to_one",
)

if ergebnisse_neu["_bewertbar"].isna().any():
    raise ValueError("Bewertbarkeit konnte nicht vollständig zugeordnet werden.")

ergebnisse_neu["p_anwesend"] = np.nan

for fold, modell in sorted(modelle_v2.items()):
    maske = ergebnisse_neu["fold"].eq(fold) & ergebnisse_neu["_bewertbar"]

    if not maske.any():
        continue

    spalten = modell_spalten_v2[fold]
    fehlende_spalten = set(spalten) - set(features.columns)

    if fehlende_spalten:
        raise ValueError(f"Modellspalten fehlen: {fehlende_spalten}")

    eingabe = features.loc[maske, spalten]

    if eingabe.isna().any().any():
        raise ValueError(f"Fold {fold}: Merkmalswerte fehlen.")

    ergebnisse_neu.loc[maske, "p_anwesend"] = modell.predict_proba(eingabe)[:, 1]

ergebnisse_neu = ergebnisse_neu.drop(columns="_bewertbar")


# ============================================================
# UNBEKANNTE KOMPONENTEN ALS NICHT BEWERTET AUFFÜHREN
# ============================================================

unbekannte = vorhanden.loc[~vorhanden["komponente"].isin(komponenten)].copy()

if not unbekannte.empty:
    unbekannte = unbekannte.merge(
        fassungen[["fassung_id", "erzeugnis", "produktgewicht", "fold"]],
        on="fassung_id",
        how="left",
        validate="many_to_one",
    )
    unbekannte["vorhanden"] = 1
    unbekannte["p_anwesend"] = np.nan
    unbekannte["pruefstatus"] = "unbekannte_komponente"

    ergebnisse_neu = pd.concat(
        [ergebnisse_neu, unbekannte],
        ignore_index=True,
    )


# ============================================================
# RANGLISTE EXAKT WIE IM MAIN AUSGEBEN
# ============================================================
# Neue Vorhersagen der korrupten Datei unter dem Main-Namen bereitstellen
ergebnisse_v2 = ergebnisse_neu.copy()

ergebnisse_v2["auffaelligkeit"] = np.where(
    ergebnisse_v2["vorhanden"].eq(0),
    ergebnisse_v2["p_anwesend"],
    1 - ergebnisse_v2["p_anwesend"]
)

# "sauber" enthält bereits die bereinigte korrupte Datei
quelldaten = pd.DataFrame(sauber)

# Die im ursprünglichen Main bereits geprüften V1-Fälle laden
rangliste_original = pd.read_csv(
    SNAPSHOT / "rangliste_v2.csv",
    dtype={
        "erzeugnis": str,
        "komponente": str
    }
)

markiert = (
    rangliste_original["bereits_in_V1_geprüft"]
    .astype(str)
    .str.lower()
    .eq("true")
)

alte_spitze = rangliste_original.loc[
    markiert,
    ["erzeugnis", "fassung_id", "komponente"]
].copy()


pflicht = [
    "erzeugnis", "fassung_id", "komponente",
    "vorhanden", "p_anwesend", "auffaelligkeit"
]

assert set(pflicht).issubset(ergebnisse_v2.columns), \
    f"Fehlende Spalten: {set(pflicht) - set(ergebnisse_v2.columns)}"


quelldaten = pd.DataFrame(sauber)

def haeufigster_text(texte):
    texte = texte.fillna("").astype(str).str.strip()
    return texte.value_counts().index[0]


produktnamen = (
    quelldaten.groupby("erzeugnis")["erzeugnis_txt"]
    .agg(haeufigster_text)
)

komponentennamen = (
    quelldaten.groupby("komponente")["komponente_txt"]
    .agg(haeufigster_text)
)


rangliste_v2 = ergebnisse_v2.loc[
    ergebnisse_v2["p_anwesend"].notna(), pflicht
].copy()

rangliste_v2["Produktname"] = (
    rangliste_v2["erzeugnis"].map(produktnamen)
)

rangliste_v2["Komponentenname"] = (
    rangliste_v2["komponente"].map(komponentennamen)
)

assert rangliste_v2["Produktname"].notna().all()
assert rangliste_v2["Komponentenname"].notna().all()


rangliste_v2["pruefrichtung"] = np.where(
    rangliste_v2["vorhanden"].eq(1),
    "möglicherweise zusätzlich",
    "möglicherweise fehlend"
)

schluessel = ["erzeugnis", "fassung_id", "komponente"]

bekannte_paare = pd.MultiIndex.from_frame(
    alte_spitze[schluessel]
)

rangliste_v2["bereits_in_V1_geprüft"] = (
    pd.MultiIndex.from_frame(rangliste_v2[schluessel])
    .isin(bekannte_paare)
)

rangliste_v2 = rangliste_v2.sort_values(
    "auffaelligkeit",
    ascending=False,
    kind="stable"
)

rangliste_v2["Rang_je_Richtung"] = (
    rangliste_v2.groupby("pruefrichtung").cumcount() + 1
)

spitzenliste_v2 = rangliste_v2.copy()

separat_v2 = ergebnisse_v2.loc[
    ergebnisse_v2["p_anwesend"].isna()
].copy()



In [0]:
display(
    spitzenliste_v2[[
        "pruefrichtung",
        "erzeugnis",
        "Produktname",
        "komponente",
        "Komponentenname",
        "auffaelligkeit",
        "vorhanden",
        "bereits_in_V1_geprüft",
        "fassung_id",
        "Rang_je_Richtung",
    ]].round(4)
)

print(
    "Separat zu prüfende Paare ohne CatBoost-Wert:",
    len(separat_v2)
)

In [0]:
top = spitzenliste_v2.iloc[0]

produkt_test = top["erzeugnis"]
fassung_test = top["fassung_id"]

kontrolle = ergebnisse_v2.loc[
    ergebnisse_v2["erzeugnis"].eq(produkt_test)
    & ergebnisse_v2["fassung_id"].eq(fassung_test)
].copy()

kontrolle["Komponentenname"] = (
    kontrolle["komponente"].map(komponentennamen)
)

kontrolle = kontrolle.loc[
    kontrolle["Komponentenname"].str.contains(
        r"RUECKSTELLFEDER|DOPPELGEHAEUSE",
        case=False,
        na=False
    ),
    [
        #"erzeugnis",
        "fassung_id",
        #"komponente",
        "Komponentenname",
        "vorhanden",
        "p_anwesend",
        "auffaelligkeit",
    ]
].sort_values("auffaelligkeit", ascending=False)

display(kontrolle)

In [0]:
_lookup = ergebnisse_v2.loc[
    ergebnisse_v2["erzeugnis"].eq(produkt_test)
    & ergebnisse_v2["fassung_id"].eq(fassung_test)
].copy()
_lookup["Komponentenname"] = _lookup["komponente"].map(komponentennamen)
doppel_id = _lookup.loc[
    _lookup["Komponentenname"].eq("DOPPELGEHAEUSE"),
    "komponente"
].iloc[0]

ziel_fassung = fassungen.loc[
    fassungen["fassung_id"].eq(fassung_test)
].iloc[0]

ziel_bloecke = block_sets.loc[
    block_sets["erzeugnis"].eq(produkt_test)
    & block_sets["komponentenset"].map(
        lambda komponenten:
        komponenten == ziel_fassung["komponentenset"]
    ),
    "block"
].tolist()

sauber_df = pd.DataFrame(sauber)

doppel_pruefung = sauber_df.loc[
    sauber_df["erzeugnis"].eq(produkt_test)
    & sauber_df["block"].isin(ziel_bloecke)
    & (
        sauber_df["komponente"].eq(doppel_id)
        | sauber_df["komponente_txt"].astype(str).str.contains(
            "Doppel", case=False, na=False
        )
    ),
    [
        "erzeugnis",
        "block",
        "komponente",
        "komponente_txt",
        "zeile_excel",
    ]
].copy()

doppel_pruefung["ist_exakt_die_erwartete_ID"] = (
    doppel_pruefung["komponente"].eq(doppel_id)
)

print("Erwartete Komponenten-ID:", doppel_id)
print("Blöcke der Fassung 1244:", ziel_bloecke)
display(doppel_pruefung)